# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a FAIR² Croissant dataset using the `mlcroissant` library. All dataset schema elements (record sets, fields, columns, etc.) are referenced by their Croissant `@id`.

### Dataset Source
The dataset schema is provided via the Croissant schema URL below.

In [ ]:
# Ensure the required library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. We use the Croissant schema URL and display main dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL (always use the latest science/doi URL)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset Croissant object
dataset = mlc.Dataset(croissant_url)  # The .metadata object, ds.records(), etc.

# Show basic information: title and description
meta = dataset.metadata
print(f"Title: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview
List available record sets and their Croissant `@id`s. Then, for each record set, print available fields (use field `@id`s).

In [ ]:
# Helper: Retrieve available record sets via Croissant metadata
def list_recordsets(meta):
    if hasattr(meta, 'record_sets'):
        record_sets = meta.record_sets
    elif hasattr(meta, 'record_set'):
        # Sometimes it's record_set
        record_sets = meta.record_set
    else:
        record_sets = []
    if hasattr(record_sets, '__iter__'):
        return list(record_sets)
    elif record_sets is not None:
        return [record_sets]
    else:
        return []

record_sets = list_recordsets(meta)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id if hasattr(rs, 'id') else rs['@id']} | Name: {getattr(rs, 'name', 'N/A')}")

    # List fields for each record set
    # Try .fields (Croissant API is flexible)
    fields = []
    if hasattr(rs, 'fields'):
        fields = rs.fields
    elif hasattr(rs, 'field'):
        fields = rs.field
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - Field @id: {f.id if hasattr(f, 'id') else f['@id']} | Name: {getattr(f, 'name', 'N/A')}")
    else:
        print("  (No fields listed for this record set)")

## 3. Data Extraction
Extract the data from the key survey record set using its Croissant `@id`, as given above. The resulting `DataFrame` will use field `@id`s as column headers, ensuring robust column references.


In [ ]:
# You can adapt these IDs to match the output above for the primary record set as needed
# For this dataset, typically the single main record set is the survey data

# --- Please adjust the RecordSet ID and Field IDs below as printed by the cell above. ---
# For demonstration, let's provide an example RecordSet @id:

# Example (replace with actual from section 2!):
main_record_set_id = 'cr:RecordSet:survey'    # Update if different

# If your record set is something like 'cr:RecordSet:kilifi_survey', use that value.

all_record_set_ids = [main_record_set_id]

dataframes = {}

# The field IDs will be something like 'cr:Field:age', 'cr:Field:phq9_score', etc.

for rs_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set @id '{rs_id}'")
        print("Columns available (field @id):")
        print(df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Error loading record set {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's perform some data cleaning and simple EDA on a numeric mental health field, e.g. PHQ-9 score (`cr:Field:phq9_score`).
- Filter for high scores (indicative of depression)
- Normalize values
- Group/aggregate by an attribute such as gender (`cr:Field:gender`) or education

> All field references use their `@id` for robustness!

In [ ]:
# Select the DataFrame from above (by RecordSet @id)
rs_id = main_record_set_id
df = dataframes[rs_id].copy()

# Specify a numeric field (by @id)
phq9_field_id = 'cr:Field:phq9_score'  # Example, change to match your overview

if phq9_field_id in df.columns:
    # Remove rows with missing PHQ-9 score
    numeric_series = pd.to_numeric(df[phq9_field_id], errors='coerce')
    df = df.assign(**{phq9_field_id: numeric_series})
    filtered_df = df[df[phq9_field_id] > 10]  # Moderate/severe
    print(f"Filtered records with {phq9_field_id} > 10 (moderate/severe): {len(filtered_df)} found.")

    # Normalize PHQ-9
    filtered_df[phq9_field_id + '_z'] = (filtered_df[phq9_field_id] - filtered_df[phq9_field_id].mean()) / filtered_df[phq9_field_id].std()
    print(filtered_df[[phq9_field_id, phq9_field_id + '_z']].head())

    # Group by gender if that column exists
    group_field_id = 'cr:Field:gender'  # Use field @id
    if group_field_id in df.columns:
        group_df = filtered_df.groupby(group_field_id)[phq9_field_id].mean().reset_index()
        print("Mean PHQ-9 score by gender (for PHQ-9 > 10):")
        print(group_df)
else:
    print(f"Field '{phq9_field_id}' not found! Please list columns above and edit this cell with a valid field @id.")

## 5. Visualization
Let's visualize the PHQ-9 score distribution (by gender, if available) using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

if phq9_field_id in df.columns:
    if 'cr:Field:gender' in df.columns:
        ax = sns.histplot(data=df, x=phq9_field_id, hue='cr:Field:gender', multiple='stack', bins=15)
        plt.title('PHQ-9 Score Distribution by Gender')
    else:
        ax = sns.histplot(df[phq9_field_id].dropna(), bins=15)
        plt.title('PHQ-9 Score Distribution')
    plt.xlabel('PHQ-9 Score (cr:Field:phq9_score)')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("PHQ-9 field not found for plotting!")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-FAIR² dataset referencing all data entities by their Croissant `@id`. This approach ensures reproducibility and robustness for cross-schema analyses. We:
- Inspected record sets and fields (@id usage)
- Loaded survey data into a DataFrame
- Filtered, normalized, and grouped PHQ-9 scores
- Visualized survey outcomes

For deeper analysis, explore additional fields (e.g. GAD-7, PSQ) or join other record sets present in the schema.